## Subject Summary Statistics by Group

In [ ]:
library(dplyr)
library(tidyr)
library(knitr)

tbl_print <- function(x) {
  dims <- paste0(nrow(x), " × ", ncol(x))
  knitr::kable(as.data.frame(x), caption = paste("A tibble:", dims))
}

In [ ]:
pheno_cleaned <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info", "visit_info.csv"))
pheno_cleaned  <- pheno_cleaned  %>% mutate(Group = factor(Group, levels = c("Healthy control", "Pre-symptomatic", "Phenoconverter","Pre-hospital", "Clinically manifest ALS")))


In [ ]:
# ── 1. Number of participants ────────────────────────────────────────────────
n_participants <- pheno_cleaned %>%
  group_by(Group) %>%
  summarise(N_participants = n_distinct(eid), .groups = "drop")

tbl_print(n_participants)

In [ ]:
# ── 2. Sex: number (%) male ──────────────────────────────────────────────────
sex_summary <- pheno_cleaned %>%
  distinct(eid, .keep_all = TRUE) %>%          # one row per participant
  group_by(Group) %>%
  summarise(
    N_male     = sum(Sex == "Male", na.rm = TRUE),
    N_total    = n(),
    Pct_male   = round(100 * N_male / N_total, 1),
    .groups = "drop"
  )

tbl_print(sex_summary)

In [ ]:
# ── 3. Baseline age (CollAge at first visit per participant) ─────────────────
baseline_age <- pheno_cleaned %>%
  arrange(eid, CollAge) %>%
  distinct(eid, .keep_all = TRUE) %>%          # keep first (earliest) visit
  group_by(Group) %>%
  summarise(
    Age_mean   = round(mean(CollAge, na.rm = TRUE), 1),
    Age_sd     = round(sd(CollAge,   na.rm = TRUE), 1),
    Age_median = round(median(CollAge, na.rm = TRUE), 1),
    Age_IQR_lo = round(quantile(CollAge, 0.25, na.rm = TRUE), 1),
    Age_IQR_hi = round(quantile(CollAge, 0.75, na.rm = TRUE), 1),
    Age_min    = round(min(CollAge, na.rm = TRUE), 1),
    Age_max    = round(max(CollAge, na.rm = TRUE), 1),
    .groups = "drop"
  )

tbl_print(baseline_age)

In [ ]:
# ── 4. Genotype group counts ─────────────────────────────────────────────────
geno_levels <-  c("SOD1 nonA4V", "C9orf72","Other Genotype","None identified")

geno_summary <- pheno_cleaned %>%
  distinct(eid, .keep_all = TRUE) %>%
  group_by(Group, GenoGroup) %>%
  summarise(N = n(), .groups = "drop") %>%
  pivot_wider(names_from = GenoGroup, values_from = N, values_fill = 0)

# Ensure all expected genotype columns are present
for (g in geno_levels) {
  if (!g %in% colnames(geno_summary)) geno_summary[[g]] <- 0
}

geno_summary <- geno_summary %>%
  select(Group, all_of(geno_levels))

tbl_print(geno_summary)

In [ ]:
# ── 5. Years from onset to baseline (YrSinceDi at first visit) ───────────────
# YrSinceDi = years since diagnosis; baseline = first visit per participant
onset_to_baseline <- pheno_cleaned %>%
  arrange(eid, CollAge) %>%
  distinct(eid, .keep_all = TRUE) %>%
  filter(!is.na(YrSinceDi)) %>%
  group_by(Group) %>%
  summarise(
    N_with_onset   = n(),
    YrSinceDi_mean = round(mean(YrSinceDi, na.rm = TRUE), 2),
    YrSinceDi_sd   = round(sd(YrSinceDi,   na.rm = TRUE), 2),
    YrSinceDi_median = round(median(YrSinceDi, na.rm = TRUE), 2),
    YrSinceDi_IQR_lo = round(quantile(YrSinceDi, 0.25, na.rm = TRUE), 2),
    YrSinceDi_IQR_hi = round(quantile(YrSinceDi, 0.75, na.rm = TRUE), 2),
    YrSinceDi_min = min(YrSinceDi, na.rm = TRUE),
    YrSinceDi_max = max(YrSinceDi, na.rm = TRUE),
    .groups = "drop"
  )

tbl_print(onset_to_baseline)

In [ ]:
# ── 6. Number of participants with longitudinal visits (>1 visit) ────────────
visit_counts_per_person <- pheno_cleaned %>%
  group_by(eid, Group) %>%
  summarise(n_visits = n(), .groups = "drop")

longitudinal <- visit_counts_per_person %>%
  group_by(Group) %>%
  summarise(
    N_longitudinal = sum(n_visits > 1),
    .groups = "drop"
  )

tbl_print(longitudinal)

In [ ]:
# ── 7. Total number of visits ────────────────────────────────────────────────
total_visits <- pheno_cleaned %>%
  group_by(Group) %>%
  summarise(Total_visits = n(), .groups = "drop")

tbl_print(total_visits)

In [ ]:
# ── 8. Number of visits per person ───────────────────────────────────────────
visits_per_person <- visit_counts_per_person %>%
  group_by(Group) %>%
  summarise(
    Visits_mean   = round(mean(n_visits), 1),
    Visits_sd     = round(sd(n_visits), 1),
    Visits_median = median(n_visits),
    Visits_IQR_lo = quantile(n_visits, 0.25),
    Visits_IQR_hi = quantile(n_visits, 0.75),
    Visits_min    = min(n_visits),
    Visits_max    = max(n_visits),
    .groups = "drop"
  )

tbl_print(visits_per_person)

In [ ]:
# ── 9. Follow-up duration (years) ────────────────────────────────────────────
# Follow-up = max(CollAge) - min(CollAge) per participant
followup <- pheno_cleaned %>%
  group_by(eid, Group) %>%
  mutate(followup_yrs = YrSinceCen) %>%
  summarise(
    followup_yrs = max(followup_yrs),
    .groups = "drop"
  ) %>%
  group_by(Group) %>%
  summarise(
    Followup_mean   = round(mean(followup_yrs), 2),
    Followup_sd     = round(sd(followup_yrs), 2),
    Followup_median = round(median(followup_yrs), 2),
    Followup_IQR_lo = round(quantile(followup_yrs, 0.25), 2),
    Followup_IQR_hi = round(quantile(followup_yrs, 0.75), 2),
    Followup_min    = min(followup_yrs),
    Followup_max    = max(followup_yrs),
    .groups = "drop"
  )

tbl_print(followup)

In [ ]:
# ── 10. Combined summary table ───────────────────────────────────────────────
summary_table <- n_participants %>%
  left_join(sex_summary %>% select(Group, N_male, Pct_male), by = "Group") %>%
  left_join(baseline_age %>% select(Group, Age_mean, Age_sd, Age_median,
                                     Age_IQR_lo, Age_IQR_hi, Age_min, Age_max), by = "Group") %>%
  left_join(geno_summary, by = "Group") %>%
  left_join(onset_to_baseline %>% select(Group, N_with_onset, YrSinceDi_mean, YrSinceDi_sd,
                                          YrSinceDi_median, YrSinceDi_IQR_lo, YrSinceDi_IQR_hi,
                                          YrSinceDi_min, YrSinceDi_max), by = "Group") %>%
  left_join(longitudinal, by = "Group") %>%
  left_join(total_visits, by = "Group") %>%
  left_join(visits_per_person %>% select(Group, Visits_mean, Visits_sd,
                                          Visits_median, Visits_IQR_lo, Visits_IQR_hi), by = "Group") %>%
  left_join(followup %>% select(Group, Followup_mean, Followup_sd, Followup_median,
                                 Followup_IQR_lo, Followup_IQR_hi, Followup_min, Followup_max), by = "Group")


In [ ]:
# ── 11. Publication-style formatted table ────────────────────────────────────
# Format: mean ± SD or median (min, max) depending on preference

pub_table <- summary_table %>%
  transmute(
    Group                                  = Group,
    `N participants`                       = N_participants,
    `Male, n (%)`                          = paste0(N_male, " (", Pct_male, "%)"),
    `Baseline age, mean ± SD`              = paste0(Age_mean, " ± ", Age_sd),
    `Baseline age, median (min, max)`      = paste0(Age_median, " (", Age_min, ", ", Age_max, ")"),
    # `SOD1 A4V, n`                        = `SOD1 A4V`,
    `SOD1 non-A4V, n`                      = `SOD1 nonA4V`,
    `C9orf72, n`                           = C9orf72,
    `Other genes, n`                       = `Other Genotype`,
    `None identified, n`                   = `None identified`,
    `Yrs onset→baseline, mean ± SD`        = ifelse(!is.na(YrSinceDi_mean),
                                                     paste0(YrSinceDi_mean, " ± ", YrSinceDi_sd), "N/A"),
    `Yrs onset→baseline, median (min, max)` = ifelse(!is.na(YrSinceDi_median),
                                                      paste0(YrSinceDi_median, " (",
                                                             round(YrSinceDi_min, 2), ", ",
                                                             round(YrSinceDi_max, 2), ")"), "N/A"),
    # `N with longitudinal visits`         = N_longitudinal,
    `Total visits`                         = Total_visits,
    # `Visits/person, mean ± SD`           = paste0(Visits_mean, " ± ", Visits_sd),
    # `Visits/person, median (IQR)`        = paste0(Visits_median, " (", Visits_IQR_lo, "–", Visits_IQR_hi, ")"),
    `Follow-up, mean ± SD (yrs)`           = paste0(Followup_mean, " ± ", Followup_sd),
    `Follow-up, median (min, max) (yrs)`   = paste0(Followup_median, " (", round(Followup_min, 2), ", ",
                                                     round(Followup_max, 2), ")")
  )

# Transpose for easy reading
pub_table_t <- as.data.frame(t(pub_table[, -1]))
colnames(pub_table_t) <- pub_table$Group
rownames(pub_table_t) <- colnames(pub_table)[-1]

tbl_print(pub_table_t)

In [ ]:
# ── 12. (Optional) Save to CSV ───────────────────────────────────────────────
# write.csv(pub_table_t, here::here("results", "subject_summary.csv"))